# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:**



**Respuesta:** 1024 valores. `Flatten()` toma la imagen de 32×32 y la estira en una sola fila, así que
multiplica las dos dimensiones: 32 × 32 = 1024. Es un solo número por píxel porque ya convertimos a
escala de grises; si siguiera en color serían 32 × 32 × 3 = 3072.

## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())

In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()

In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:**


**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:**


**4.** ¿Qué función cumple `ReLU`?

**Respuesta:**



**2.** Porque `Dense` (capa donde cada neurona se conecta con todas las entradas) solo acepta un vector
plano, no una matriz. La imagen llega como 32×32 y `Flatten()` la convierte en una lista de 1024 números.

**3.** Porque CIFAR-10 tiene 10 clases y la red produce un valor por clase. Cada neurona de salida
entrega el puntaje de una clase, y la predicción es la de mayor puntaje.

**4.** `ReLU` es la función que convierte en 0 todo valor negativo y deja igual los positivos. Aporta
**no linealidad**: sin ella, apilar capas `Dense` daría lo mismo que una sola capa, y la red no podría
aprender relaciones complejas.

---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:**



**Respuesta:** Porque la última capa `Dense(10)` no tiene activación, así que produce **logits**
(puntajes sin normalizar, que pueden ser negativos o mayores que 1) y no probabilidades. Con
`from_logits=True` le avisamos a la función de pérdida que ella misma debe aplicar el Softmax
internamente. Hacerlo así es más estable numéricamente que aplicar Softmax en la capa y luego calcular
el logaritmo por separado.

---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)

### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:**


**7.** ¿Qué representa `batch_size=64`?

**Respuesta:**



**6.** Una `epoch` es una pasada completa por todas las imágenes de entrenamiento. Con 10 epochs, la red
recorre las 50.000 imágenes diez veces, ajustando sus pesos en cada recorrido.

**7.** `batch_size=64` significa que los pesos se actualizan cada 64 imágenes, no una por una ni con
todas a la vez. Cada actualización usa el promedio del error de ese grupo, lo que da un entrenamiento
más rápido y estable que ir de a una.

## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:**



**Respuesta:** Compara las dos curvas de la gráfica anterior. Hay overfitting (la red memoriza el
entrenamiento en vez de generalizar) si el accuracy de entrenamiento sigue subiendo mientras el de
validación se estanca o baja, y la separación entre ambas curvas crece con las epochs. Si las dos
suben juntas y quedan cerca, no hay evidencia clara. Escribe aquí qué observaste y en qué epoch
empezaron a separarse.

---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")

### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:**



**Respuesta:** Anota tu accuracy de prueba y compáralo con el de Fashion-MNIST visto en clase.
CIFAR-10 en escala de grises es **más difícil** para esta red: sus imágenes son fotografías reales con
fondos, ángulos y texturas variadas, mientras que Fashion-MNIST son prendas centradas sobre fondo negro
y bien recortadas. Además, al pasar a grises perdimos el color, que era una pista útil para separar
clases como *ship* o *frog*.

---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])

In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:**



**Respuesta:** Porque Softmax divide cada valor exponenciado entre la suma de todos los valores
exponenciados. Al dividir cada término por ese mismo total, la suma de los diez resultados da 1 por
construcción. Eso es justo lo que permite leerlos como probabilidades: se reparte el 100% de la
confianza entre las 10 clases. En la práctica puede dar 0.9999999 por redondeo del computador.

---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta:**


### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:**


### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante sobre la imagen podría perderse al tratarla únicamente como un vector?

**Respuesta:**


### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial de las imágenes?

**Respuesta:**

---

## Para pensar

Nuestro modelo hace:

```text
32×32 → Flatten → 1024 valores → Dense → ReLU → Dense(10)
```

Sin embargo, una imagen tiene **estructura espacial**: existen bordes, formas y patrones locales.

Este resultado nos deja preparada la siguiente pregunta del curso:

> **¿Cómo puede una red aprovechar directamente la estructura espacial de una imagen?**

La respuesta será: **Convolutional Neural Networks (CNNs)**.


## Respuestas — Reflexión final

**1.** Escribe aquí el valor que imprimió la celda de evaluación (`Accuracy (%)`).

**2.** Revisa la cuadrícula de predicciones de la sección 7 y anota qué pares se confunden. Es esperable
que las clases más parecidas en silueta se mezclen (por ejemplo los animales de cuatro patas entre sí, o
*automobile* con *truck*), sobre todo ahora que no hay color para distinguirlas.

**3.** Se pierde la **estructura espacial**: cuáles píxeles eran vecinos, dónde había bordes y formas.
Para la red, el vector de 1024 números no tiene arriba ni abajo; si desordenáramos siempre igual todos
los píxeles, aprendería exactamente lo mismo. Tampoco reconoce un objeto si aparece desplazado.

**4.** Una **red convolucional (CNN)**, que aplica filtros pequeños sobre regiones vecinas de la imagen
y por eso sí aprovecha la cercanía entre píxeles y detecta patrones sin importar dónde aparezcan.